# 02 — Tensors and Operations

## Learning Objectives

By the end of this notebook, you should be able to:

- Understand tensor dimensions and axes deeply.
- Perform element-wise and matrix operations.
- Understand broadcasting and why it works.
- Use `dim` and `keepdim` correctly.
- Concatenate and stack tensors.
- Reshape, view, flatten, squeeze, and unsqueeze tensors.
- Understand transpose, permutation, and contiguous memory.
- Perform comparison, logical, and mathematical operations.
- Work with tensor statistics and normalization.
- Understand common tensor-shape errors.

> **Teaching approach:** For important parameters, we change them, observe the output, and explain **why** the output changed.

# 1. Tensor Operations

A tensor operation takes one or more tensors and produces another tensor.

Examples:

```python
a + b
a * b
a @ b
torch.sum(a)
a.reshape(...)
```

In deep learning, almost everything eventually becomes tensor operations:

```text
Input → Tensor operations → Model → Prediction → Loss → Gradients → Update
```

In [84]:
import torch

print("PyTorch version:", torch.__version__)

PyTorch version: 2.9.1


# 2. Element-wise Operations

Element-wise operations operate on corresponding elements.

```text
A = [1, 2, 3]
B = [10, 20, 30]

A + B = [11, 22, 33]
```

Common operations:

- `+`
- `-`
- `*`
- `/`
- `**`

In [85]:
a = torch.tensor([1., 2., 3.])
b = torch.tensor([10., 20., 30.])

print("Addition:", a + b)
print("Subtraction:", a - b)
print("Multiplication:", a * b)
print("Division:", a / b)
print("Power:", a ** 2)

Addition: tensor([11., 22., 33.])
Subtraction: tensor([ -9., -18., -27.])
Multiplication: tensor([10., 40., 90.])
Division: tensor([0.1000, 0.1000, 0.1000])
Power: tensor([1., 4., 9.])


## `*` versus `@`

```python
A * B
```

means **element-wise multiplication**.

```python
A @ B
```

means **matrix multiplication**.

In [86]:
A = torch.tensor([[1., 2.], [3., 4.]])
B = torch.tensor([[5., 6.], [7., 8.]])

print("Element-wise:")
print(A * B)

print("\nMatrix multiplication:")
print(A @ B)

Element-wise:
tensor([[ 5., 12.],
        [21., 32.]])

Matrix multiplication:
tensor([[19., 22.],
        [43., 50.]])


# 3. Scalar Operations

A scalar is applied to every element.

This is also an example of broadcasting.

In [87]:
x = torch.tensor([1., 2., 3., 4.])

print(x + 10)
print(x * 10)
print(x / 10)

tensor([11., 12., 13., 14.])
tensor([10., 20., 30., 40.])
tensor([0.1000, 0.2000, 0.3000, 0.4000])


# 4. Broadcasting

Broadcasting allows compatible tensors with different shapes to participate in an operation.

Example:

```text
A → (2, 3)
B → (3,)
```

PyTorch can apply `B` across every row of `A`.

In [88]:
A = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])

B = torch.tensor([10., 20., 30.])

print(A + B)
print("Result shape:", (A + B).shape)

tensor([[11., 22., 33.],
        [14., 25., 36.]])
Result shape: torch.Size([2, 3])


## Broadcasting rules

PyTorch compares dimensions from **right to left**.

Two dimensions are compatible when:

1. They are equal, or
2. One is `1`, or
3. One tensor does not have that dimension.

Example:

```text
(2, 3)
(3,)
```

The final `3` matches, so the operation works.

## Broadcasting with dimension 1

In [89]:
A = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])

B = torch.tensor([[10.],
                  [20.]])

print("A shape:", A.shape)
print("B shape:", B.shape)
print("Result:")
print(A + B)

A shape: torch.Size([2, 3])
B shape: torch.Size([2, 1])
Result:
tensor([[11., 12., 13.],
        [24., 25., 26.]])


Conceptually `B` behaves like:

```text
[10, 10, 10]
[20, 20, 20]
```

so its effective shape becomes compatible with `(2, 3)`.

## Broadcasting error

```text
(2, 3)
(2, 2)
```

fails because the last dimensions are `3` and `2`, and neither is `1`.

In [90]:
A = torch.ones(2, 3)
B = torch.ones(2, 2)

try:
    print(A + B)
except RuntimeError as e:
    print("Expected error:")
    print(e)

Expected error:
The size of tensor a (3) must match the size of tensor b (2) at non-singleton dimension 1


# 5. Matrix Multiplication

For 2D matrices:

```text
A → (m, n)
B → (n, p)

A @ B → (m, p)
```

The **inner dimensions must match**.

In [91]:
A = torch.randn(2, 3)
B = torch.randn(3, 4)

C = A @ B

print("A:", A.shape)
print("B:", B.shape)
print("C:", C.shape)

A: torch.Size([2, 3])
B: torch.Size([3, 4])
C: torch.Size([2, 4])


Here:

```text
(2, 3) @ (3, 4) → (2, 4)
       ↑
   must match
```

This operation is fundamental to linear layers and many other neural-network computations.

In [92]:
A = torch.randn(2, 3)
B = torch.randn(2, 4)

try:
    print(A @ B)
except RuntimeError as e:
    print("Expected matrix multiplication error:")
    print(e)

Expected matrix multiplication error:
mat1 and mat2 shapes cannot be multiplied (2x3 and 2x4)


# 6. `matmul()`, `mm()`, and `@`

For ordinary 2D matrices:

```python
A @ B
torch.matmul(A, B)
torch.mm(A, B)
```

can perform matrix multiplication.

`torch.matmul()` is more general and supports higher-dimensional batched operations.

In [93]:
A = torch.randn(3, 4)
B = torch.randn(4, 2)

r1 = A @ B
r2 = torch.matmul(A, B)
r3 = torch.mm(A, B)

print(torch.allclose(r1, r2))
print(torch.allclose(r1, r3))

True
True


# 7. Dot Product

For vectors:

```text
[1, 2, 3] · [10, 20, 30]
= 1×10 + 2×20 + 3×30
= 140
```

In [94]:
a = torch.tensor([1., 2., 3.])
b = torch.tensor([10., 20., 30.])

print("dot:", torch.dot(a, b))
print("@:", a @ b)

dot: tensor(140.)
@: tensor(140.)


# 8. Transpose

For a 2D tensor, transpose swaps rows and columns.

```text
(2, 3) → (3, 2)
```

In [95]:
x = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])

print("Original:")
print(x)
print("Shape:", x.shape)

print("\nTranspose:")
print(x.T)
print("Shape:", x.T.shape)

Original:
tensor([[1, 2, 3],
        [4, 5, 6]])
Shape: torch.Size([2, 3])

Transpose:
tensor([[1, 4],
        [2, 5],
        [3, 6]])
Shape: torch.Size([3, 2])


## `transpose(dim0, dim1)`

Syntax:

```python
tensor.transpose(dim0, dim1)
```

It swaps exactly two dimensions.

In [96]:
x = torch.randn(2, 3, 4)

print("Original:", x.shape)

y = x.transpose(0, 1)

print("transpose(0, 1):", y.shape)

Original: torch.Size([2, 3, 4])
transpose(0, 1): torch.Size([3, 2, 4])


The shape changes:

```text
(2, 3, 4)
→ (3, 2, 4)
```

because dimensions `0` and `1` were exchanged.

# 9. `permute()`

`permute()` reorders **all dimensions**.

Syntax:

```python
tensor.permute(dim0, dim1, ...)
```

In [97]:
x = torch.randn(2, 3, 4)

print("Original:", x)

print("Original:", x.shape)

y = x.permute(2, 0, 1)

print("Permuted:", y)

print("Permuted:", y.shape)

Original: tensor([[[ 0.1782,  0.2619, -0.8980, -0.3782],
         [-1.0548, -0.4414, -1.0109,  0.2290],
         [ 0.2515, -0.4424, -1.4110,  0.1198]],

        [[ 0.2239,  0.3430, -1.0977,  0.6239],
         [ 0.7321, -1.3998, -0.7414,  0.1191],
         [-0.0871, -1.1957, -2.1299, -2.0019]]])
Original: torch.Size([2, 3, 4])
Permuted: tensor([[[ 0.1782, -1.0548,  0.2515],
         [ 0.2239,  0.7321, -0.0871]],

        [[ 0.2619, -0.4414, -0.4424],
         [ 0.3430, -1.3998, -1.1957]],

        [[-0.8980, -1.0109, -1.4110],
         [-1.0977, -0.7414, -2.1299]],

        [[-0.3782,  0.2290,  0.1198],
         [ 0.6239,  0.1191, -2.0019]]])
Permuted: torch.Size([4, 2, 3])


The dimension order changes from:

```text
(2, 3, 4)
```

to:

```text
(4, 2, 3)
```

because we requested:

```python
(2, 0, 1)
```

# 10. Image Dimension Example

A single image is often represented in PyTorch as:

```text
(C, H, W)
```

For RGB `224 × 224`:

```text
(3, 224, 224)
```

A batch is:

```text
(N, C, H, W)
```

In [98]:
image = torch.rand(224, 224, 3)  # H, W, C

print("HWC:", image.shape)

image_chw = image.permute(2, 0, 1)

print("CHW:", image_chw.shape)

batch = image_chw.unsqueeze(0)

print("NCHW:", batch.shape)

HWC: torch.Size([224, 224, 3])
CHW: torch.Size([3, 224, 224])
NCHW: torch.Size([1, 3, 224, 224])


The shape flow is:

```text
(H, W, C)
    ↓ permute
(C, H, W)
    ↓ unsqueeze(0)
(N, C, H, W)
```

This is a very common operation when preparing image data for PyTorch models.

# 11. `reshape()`

`reshape()` changes the shape while keeping the same number of elements.

Syntax:

```python
tensor.reshape(new_shape)
```

The product of the new dimensions must equal the original number of elements.

In [99]:
x = torch.arange(1, 7)

print("Original:", x)
print("Original shape:", x.shape)

y = x.reshape(2, 3)

print("\nReshaped:")
print(y)
print("Shape:", y.shape)

Original: tensor([1, 2, 3, 4, 5, 6])
Original shape: torch.Size([6])

Reshaped:
tensor([[1, 2, 3],
        [4, 5, 6]])
Shape: torch.Size([2, 3])


The tensor contains 6 elements.

Therefore these are valid:

```text
(2, 3) → 6
(3, 2) → 6
(1, 6) → 6
(6,)   → 6
```

But `(4, 2)` contains 8 elements and is invalid.

In [100]:
x = torch.arange(1, 7)

try:
    print(x.reshape(4, 2))
except RuntimeError as e:
    print("Expected error:")
    print(e)

Expected error:
shape '[4, 2]' is invalid for input of size 6


# 12. `reshape()` vs `view()`

Both can change shape.

`view()` generally requires a compatible memory layout.

`reshape()` is more flexible: it can return a view when possible or create a copy when necessary.

For simple shape changes, `reshape()` is usually the easier choice.

In [101]:
x = torch.arange(12)

print("reshape:")
print(x.reshape(3, 4))

print("\nview:")
print(x.view(3, 4))

reshape:
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

view:
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])


# 13. Contiguous Memory

Check memory layout with:

```python
tensor.is_contiguous()
```

Operations such as transpose or permute can produce a non-contiguous view.

In [102]:
x = torch.arange(12).reshape(3, 4)

print("Original contiguous:", x.is_contiguous())

y = x.transpose(0, 1)

print("Transposed contiguous:", y.is_contiguous())

z = y.contiguous()

print("After contiguous():", z.is_contiguous())

Original contiguous: True
Transposed contiguous: False
After contiguous(): True


`contiguous()` creates a tensor with a standard contiguous memory layout.

This can be useful before operations that require compatible memory layout, such as certain uses of `view()`.

# 14. `flatten()`

Syntax:

```python
torch.flatten(input, start_dim=0, end_dim=-1)
```

Important parameters:

- `input` → tensor
- `start_dim` → first dimension to flatten
- `end_dim` → last dimension to flatten

In [103]:
x = torch.randn(2, 3, 4)

print("Original:", x.shape)
print("Flatten all:", x.flatten().shape)
print("Flatten from dim 1:", x.flatten(start_dim=1).shape)

Original: torch.Size([2, 3, 4])
Flatten all: torch.Size([24])
Flatten from dim 1: torch.Size([2, 12])


A common CNN example:

```text
(32, 64, 7, 7)
       ↓ flatten(start_dim=1)
(32, 3136)
```

because:

```text
64 × 7 × 7 = 3136
```

Notice that the batch dimension `32` is preserved.

# 15. `unsqueeze()`

Adds a dimension of size `1`.

```python
tensor.unsqueeze(dim)
```

In [104]:
x = torch.tensor([10., 20., 30.])

print("Original:", x.shape)
print("unsqueeze(0):", x.unsqueeze(0).shape)
print("unsqueeze(1):", x.unsqueeze(1).shape)
print("unsqueeze(1):", x.unsqueeze(-1).shape)

Original: torch.Size([3])
unsqueeze(0): torch.Size([1, 3])
unsqueeze(1): torch.Size([3, 1])
unsqueeze(1): torch.Size([3, 1])


The `dim` parameter controls **where** the new dimension is inserted.

# 16. `squeeze()`

Removes dimensions of size `1`.

```python
tensor.squeeze(dim=None)
```

If `dim` is omitted, all size-1 dimensions may be removed.

In [105]:
x = torch.randn(1, 3, 1, 5)

print("Original:", x.shape)
print("squeeze():", x.squeeze().shape)
print("squeeze(0):", x.squeeze(0).shape)

Original: torch.Size([1, 3, 1, 5])
squeeze(): torch.Size([3, 5])
squeeze(0): torch.Size([3, 1, 5])


### ⚠️ Important

If the batch size happens to be `1`, an unrestricted `squeeze()` can accidentally remove your batch dimension.

When the intended dimension is known, specifying `dim` is safer.

# 17. Reduction Operations

Reduction operations summarize values.

Common functions:

- `sum()`
- `mean()`
- `min()`
- `max()`
- `prod()`
- `std()`
- `var()`

In [106]:
x = torch.tensor([1., 2., 3., 4.])

print("sum:", x.sum())
print("mean:", x.mean())
print("min:", x.min())
print("max:", x.max())
print("product:", x.prod())
print("std:", x.std())
print("variance:", x.var())

sum: tensor(10.)
mean: tensor(2.5000)
min: tensor(1.)
max: tensor(4.)
product: tensor(24.)
std: tensor(1.2910)
variance: tensor(1.6667)


# 18. The `dim` Parameter

For a matrix:

```text
1 2 3
4 5 6
```

`dim=0` operates down the rows.

`dim=1` operates across the columns.

In [107]:
x = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])

print("sum(dim=0):", x.sum(dim=0))
print("sum(dim=1):", x.sum(dim=1))

sum(dim=0): tensor([5., 7., 9.])
sum(dim=1): tensor([ 6., 15.])


### Shapes

```text
x.shape             = (2, 3)

x.sum(dim=0).shape  = (3,)
x.sum(dim=1).shape  = (2,)
```

The selected dimension disappears by default.

# 19. `keepdim=True`

Normally a reduction removes the reduced dimension.

```text
(2, 3)
sum(dim=1)
→ (2,)
```

With `keepdim=True`:

```text
(2, 3)
sum(dim=1, keepdim=True)
→ (2, 1)
```

In [108]:
x = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])

a = x.sum(dim=1)
b = x.sum(dim=1, keepdim=True)

print("Without keepdim:", a, a.shape)
print("\nWith keepdim:", b, b.shape)

Without keepdim: tensor([ 6., 15.]) torch.Size([2])

With keepdim: tensor([[ 6.],
        [15.]]) torch.Size([2, 1])


`keepdim=True` is useful when the reduced result needs to participate in later broadcasting.

# 20. `argmax()` and `argmin()`

`argmax()` returns the index of the largest value.

`argmin()` returns the index of the smallest value.

In [109]:
x = torch.tensor([10., 50., 20., 90., 30.])

print("Maximum value:", x.max())
print("Maximum index:", x.argmax())

print("Minimum value:", x.min())
print("Minimum index:", x.argmin())

Maximum value: tensor(90.)
Maximum index: tensor(3)
Minimum value: tensor(10.)
Minimum index: tensor(0)


## Classification example

If a model outputs class scores:

```text
Class 0 → 0.2
Class 1 → 2.5
Class 2 → 0.8
```

the highest score is at index `1`.

In [110]:
scores = torch.tensor([0.2, 2.5, 0.8])

print("Predicted class:", scores.argmax().item())

Predicted class: 1


For a batch, use `dim=1` when the tensor is:

```text
(batch_size, number_of_classes)
```

In [111]:
scores = torch.tensor([
    [0.2, 2.5, 0.8],
    [3.0, 0.5, 1.2],
    [0.1, 0.4, 4.2]
])

predictions = scores.argmax(dim=1)

print("Scores shape:", scores.shape)
print("Predictions:", predictions)

Scores shape: torch.Size([3, 3])
Predictions: tensor([1, 0, 2])


# 21. Comparison and Boolean Masks

Comparisons produce Boolean tensors.

```python
>
<
>=
<=
==
!=
```

In [112]:
x = torch.tensor([1, 5, 3, 8])

print("x > 3:", x > 3)
print("x < 5:", x < 5)
print("x == 5:", x == 5)
print("x != 5:", x != 5)

x > 3: tensor([False,  True, False,  True])
x < 5: tensor([ True, False,  True, False])
x == 5: tensor([False,  True, False, False])
x != 5: tensor([ True, False,  True,  True])


Boolean masks can be used to select values.

In [113]:
x = torch.tensor([1, 5, 3, 8, 2, 10])

mask = x > 4

print("Mask:", mask)
print("Selected:", x[mask])

Mask: tensor([False,  True, False,  True, False,  True])
Selected: tensor([ 5,  8, 10])


# 22. `torch.where()`

Syntax:

```python
torch.where(condition, input, other)
```

It selects `input` where the condition is true and `other` where it is false.

In [114]:
x = torch.tensor([-2., -1., 0., 1., 2.])

result = torch.where(x > 0, x, torch.tensor(0.))

print("Original:", x)
print("Result:", result)

Original: tensor([-2., -1.,  0.,  1.,  2.])
Result: tensor([0., 0., 0., 1., 2.])


This resembles the mathematical idea behind ReLU:

```text
negative → 0
positive → unchanged
```

# 23. Mathematical Functions

Useful functions include:

- `torch.abs()`
- `torch.sqrt()`
- `torch.exp()`
- `torch.log()`
- `torch.sin()`
- `torch.cos()`
- `torch.sigmoid()`

In [115]:
x = torch.tensor([-4., -1., 0., 1., 4.])

print("abs:", torch.abs(x))
print("sqrt(abs):", torch.sqrt(torch.abs(x)))
print("exp:", torch.exp(x))

abs: tensor([4., 1., 0., 1., 4.])
sqrt(abs): tensor([2., 1., 0., 1., 2.])
exp: tensor([1.8316e-02, 3.6788e-01, 1.0000e+00, 2.7183e+00, 5.4598e+01])


# 24. `clamp()`

`torch.clamp()` restricts values to a range.

Syntax:

```python
torch.clamp(input, min=None, max=None)
```

Parameters:

- `input` → tensor
- `min` → lower bound
- `max` → upper bound

In [116]:
x = torch.tensor([-10., -2., 0., 5., 20.])

print(torch.clamp(x, min=-2, max=5))

tensor([-2., -2.,  0.,  5.,  5.])


Values below `-2` become `-2`.

Values above `5` become `5`.

Values inside the range stay unchanged.

# 25. `cat()` — Concatenation

`torch.cat()` joins tensors along an **existing dimension**.

Syntax:

```python
torch.cat(tensors, dim=0)
```

Important parameters:

- `tensors` → sequence of tensors
- `dim` → dimension to concatenate along

In [117]:
a = torch.tensor([[1, 2],
                  [3, 4]])

b = torch.tensor([[5, 6],
                  [7, 8]])

c0 = torch.cat([a, b], dim=0)
c1 = torch.cat([a, b], dim=1)

print("dim=0:")
print(c0)
print("Shape:", c0.shape)

print("\ndim=1:")
print(c1)
print("Shape:", c1.shape)

dim=0:
tensor([[1, 2],
        [3, 4],
        [5, 6],
        [7, 8]])
Shape: torch.Size([4, 2])

dim=1:
tensor([[1, 2, 5, 6],
        [3, 4, 7, 8]])
Shape: torch.Size([2, 4])


```text
dim=0:
(2,2) + (2,2) → (4,2)

dim=1:
(2,2) + (2,2) → (2,4)
```

The selected dimension grows.

# 26. `stack()`

`torch.stack()` creates a **new dimension**.

Syntax:

```python
torch.stack(tensors, dim=0)
```

In [118]:
a = torch.tensor([1, 2, 3])
b = torch.tensor([4, 5, 6])

cat_result = torch.cat([a, b])
stack_result = torch.stack([a, b])

print("cat:", cat_result, cat_result.shape)
print("\nstack:")
print(stack_result)
print("Shape:", stack_result.shape)

cat: tensor([1, 2, 3, 4, 5, 6]) torch.Size([6])

stack:
tensor([[1, 2, 3],
        [4, 5, 6]])
Shape: torch.Size([2, 3])


### Remember

```text
cat   → joins an existing dimension
stack → creates a new dimension
```

# 27. `chunk()` and `split()`

`chunk()` divides a tensor into a requested number of chunks.

`split()` divides according to specified sizes.

In [119]:
x = torch.arange(12)

chunks = torch.chunk(x, 3)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i}:", chunk)

Chunk 0: tensor([0, 1, 2, 3])
Chunk 1: tensor([4, 5, 6, 7])
Chunk 2: tensor([ 8,  9, 10, 11])


In [120]:
x = torch.arange(10)

parts = torch.split(x, [3, 2, 5])

for i, part in enumerate(parts):
    print(f"Part {i}:", part)

Part 0: tensor([0, 1, 2])
Part 1: tensor([3, 4])
Part 2: tensor([5, 6, 7, 8, 9])


# 28. References vs `clone()`

This:

```python
y = x
```

does not create an independent tensor.

Use:

```python
y = x.clone()
```

when you need an independent copy of the tensor data.

In [121]:
x = torch.tensor([1., 2., 3.])
y = x

y[0] = 100

print("x:", x)
print("y:", y)

x: tensor([100.,   2.,   3.])
y: tensor([100.,   2.,   3.])


In [122]:
x = torch.tensor([1., 2., 3.])
y = x.clone()

y[0] = 100

print("x:", x)
print("y:", y)

x: tensor([1., 2., 3.])
y: tensor([100.,   2.,   3.])


# 29. `detach()`

`detach()` creates a tensor disconnected from the current autograd graph.

```python
y = x.detach()
```

We will study this in detail in Notebook 03.

In [123]:
x = torch.tensor([2.0], requires_grad=True)

y = x * 3
z = y.detach()

print("x requires_grad:", x.requires_grad)
print("y requires_grad:", y.requires_grad)
print("z requires_grad:", z.requires_grad)

x requires_grad: True
y requires_grad: True
z requires_grad: False


# 30. In-place Operations

Operations ending with `_` commonly modify the tensor in-place.

Example:

```python
x.add_(10)
```

In [124]:
x = torch.tensor([1., 2., 3.])

print("Before:", x)

x.add_(10)

print("After:", x)

Before: tensor([1., 2., 3.])
After: tensor([11., 12., 13.])


In-place operations can save memory, but they can interfere with autograd and intermediate values.

Use them carefully in neural-network code.

# 31. Tensor Statistics

Random data from `randn()` should have statistics approximately like a standard normal distribution.

In [125]:
x = torch.randn(1000)

print("Mean:", x.mean().item())
print("Std :", x.std().item())
print("Min :", x.min().item())
print("Max :", x.max().item())

Mean: -0.004559166729450226
Std : 1.0215054750442505
Min : -3.194124937057495
Max : 3.1266896724700928


The mean should be around `0` and standard deviation around `1`, but finite random samples will not be exact.

# 32. Normalization

A common normalization formula is:

```text
x_normalized = (x - mean) / std
```

In [126]:
x = torch.tensor([10., 20., 30., 40., 50.])

mean = x.mean()
std = x.std()

normalized = (x - mean) / std

print("Original:", x)
print("Mean:", mean)
print("Std:", std)
print("Normalized:", normalized)
print("New mean:", normalized.mean())
print("New std:", normalized.std())

Original: tensor([10., 20., 30., 40., 50.])
Mean: tensor(30.)
Std: tensor(15.8114)
Normalized: tensor([-1.2649, -0.6325,  0.0000,  0.6325,  1.2649])
New mean: tensor(0.)
New std: tensor(1.)


Normalization is widely used in machine learning because scaling can make optimization easier.

The exact normalization method depends on the dataset and model.

# 33. One-Hot Encoding

Class IDs:

```text
0, 2, 1
```

can be represented as:

```text
0 → [1,0,0]
1 → [0,1,0]
2 → [0,0,1]
```

In [127]:
labels = torch.tensor([0, 2, 1])

one_hot = torch.nn.functional.one_hot(
    labels,
    num_classes=3
)

print("Labels:", labels)
print("One-hot:")
print(one_hot)
print("Shape:", one_hot.shape)

Labels: tensor([0, 2, 1])
One-hot:
tensor([[1, 0, 0],
        [0, 0, 1],
        [0, 1, 0]])
Shape: torch.Size([3, 3])


`num_classes` determines the width of the one-hot representation.

For 3 classes, each row has 3 positions.

# 34. Dtype Conversion

Common conversions:

```python
.float()
.long()
.int()
```

or:

```python
.to(dtype=...)
```

In [128]:
x = torch.tensor([1, 2, 3])

print("Original:", x.dtype)
print("float:", x.float().dtype)
print("long:", x.long().dtype)
print("int:", x.int().dtype)

Original: torch.int64
float: torch.float32
long: torch.int64
int: torch.int32


Typical deep-learning convention:

```text
Input features → floating point
Class indices  → int64 / long
```

The exact requirement depends on the operation or loss function.

# 35. Device and Tensor Operations

Tensors can live on CPU or CUDA GPU.

A useful pattern is:

In [129]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Selected device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Selected device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


Move a tensor with:

```python
tensor.to(device)
```

In [130]:
x = torch.randn(3, 3)

x = x.to(device)

print("Device:", x.device)

Device: cuda:0


Operations between tensors generally require compatible devices.

A CPU tensor and CUDA tensor cannot normally be directly combined.

In [131]:
cpu_tensor = torch.tensor([1., 2., 3.])

if torch.cuda.is_available():
    gpu_tensor = torch.tensor([4., 5., 6.], device="cuda")

    try:
        print(cpu_tensor + gpu_tensor)
    except RuntimeError as e:
        print("Expected device mismatch:")
        print(e)
else:
    print("CUDA unavailable; skipped.")

Expected device mismatch:
Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!


# 36. NumPy Interoperability

CPU tensors can be converted to NumPy:

```python
tensor.numpy()
```

and NumPy arrays can be converted with:

```python
torch.from_numpy(array)
```

In [132]:
import numpy as np

array = np.array([10, 20, 30], dtype=np.float32)

tensor = torch.from_numpy(array)

print("NumPy:", array)
print("Tensor:", tensor)
print("Tensor dtype:", tensor.dtype)

NumPy: [10. 20. 30.]
Tensor: tensor([10., 20., 30.])
Tensor dtype: torch.float32


`from_numpy()` can share memory with the NumPy array.

In [133]:
array = np.array([1, 2, 3], dtype=np.float32)
tensor = torch.from_numpy(array)

tensor[0] = 999

print("Tensor:", tensor)
print("NumPy:", array)

Tensor: tensor([999.,   2.,   3.])
NumPy: [999.   2.   3.]


Because the memory can be shared, changing one can affect the other.

If you need independent storage, make a copy instead of relying on shared memory behavior.

# 37. Random Seeds

Use:

```python
torch.manual_seed(42)
```

to make random generation reproducible for the relevant PyTorch random operations.

In [134]:
torch.manual_seed(42)
a = torch.rand(5)

torch.manual_seed(42)
b = torch.rand(5)

print("a:", a)
print("b:", b)
print("Same:", torch.equal(a, b))

a: tensor([0.8823, 0.9150, 0.3829, 0.9593, 0.3904])
b: tensor([0.8823, 0.9150, 0.3829, 0.9593, 0.3904])
Same: True


Changing the seed changes the generated sequence.

The seed does not make the result better; it makes the random sequence repeatable.

# 38. Memory Usage

`numel()` gives the number of elements.

`element_size()` gives the bytes used by each element.

Approximate memory:

```text
numel × element_size
```

In [135]:
x = torch.zeros(32, 3, 224, 224)

print("Shape:", x.shape)
print("Elements:", x.numel())
print("Bytes/element:", x.element_size())
print("Approx MB:", x.numel() * x.element_size() / (1024 ** 2))

Shape: torch.Size([32, 3, 224, 224])
Elements: 4816896
Bytes/element: 4
Approx MB: 18.375


This becomes important when working with large images, batches, and neural-network models on limited GPU memory.

# 39. Neural-Network Style Broadcasting

Suppose:

```text
X → (batch, features)
bias → (features,)
```

Adding them broadcasts the bias across every sample.

This is related to the mathematical operation:

```text
Y = XW + b
```

In [136]:
X = torch.tensor([
    [1., 2., 3.],
    [4., 5., 6.],
    [7., 8., 9.]
])

bias = torch.tensor([0.5, 1.0, 2.0])

Y = X + bias

print("X shape:", X.shape)
print("Bias shape:", bias.shape)
print("Y:")
print(Y)

X shape: torch.Size([3, 3])
Bias shape: torch.Size([3])
Y:
tensor([[ 1.5000,  3.0000,  5.0000],
        [ 4.5000,  6.0000,  8.0000],
        [ 7.5000,  9.0000, 11.0000]])


# 40. Shape Debugging

When debugging PyTorch, inspect at least:

```python
tensor.shape
tensor.dtype
tensor.device
```

In [137]:
x = torch.randn(32, 3, 224, 224)

print("Shape :", x.shape)
print("Dims  :", x.ndim)
print("Dtype :", x.dtype)
print("Device:", x.device)
print("Numel :", x.numel())

Shape : torch.Size([32, 3, 224, 224])
Dims  : 4
Dtype : torch.float32
Device: cpu
Numel : 4816896


When a model reports a shape error:

1. Print the input shape.
2. Print intermediate shapes.
3. Check the expected shape of the next layer.
4. Check whether a batch dimension is missing.
5. Check `dim`, `reshape`, `permute`, and broadcasting.

# 41. Mini Experiment — `dim` and `keepdim`

Predict the output shapes before running this code.

In [138]:
x = torch.randn(2, 3, 4)

for dim in range(x.ndim):
    result = x.sum(dim=dim)
    kept = x.sum(dim=dim, keepdim=True)

    print(
        f"dim={dim} | normal={result.shape} | keepdim={kept.shape}"
    )

dim=0 | normal=torch.Size([3, 4]) | keepdim=torch.Size([1, 3, 4])
dim=1 | normal=torch.Size([2, 4]) | keepdim=torch.Size([2, 1, 4])
dim=2 | normal=torch.Size([2, 3]) | keepdim=torch.Size([2, 3, 1])


# 42. Mini Experiment — Broadcasting

Predict the shapes first.

```text
A → (3,4)
B → (4,)
B → (3,1)
B → (3,2)
```

In [139]:
A = torch.randn(3, 4)

B1 = torch.randn(4)
print("(3,4) + (4,) ->", (A + B1).shape)

B2 = torch.randn(3, 1)
print("(3,4) + (3,1) ->", (A + B2).shape)

B3 = torch.randn(3, 2)

try:
    print(A + B3)
except RuntimeError as e:
    print("\nExpected error:")
    print(e)

(3,4) + (4,) -> torch.Size([3, 4])
(3,4) + (3,1) -> torch.Size([3, 4])

Expected error:
The size of tensor a (4) must match the size of tensor b (2) at non-singleton dimension 1


# 43. Mini Experiment — Matrix Multiplication

Predict each output:

```text
(5,3) @ (3,2)
(10,8) @ (8,4)
(2,7) @ (7,1)
```

In [140]:
examples = [
    ((5, 3), (3, 2)),
    ((10, 8), (8, 4)),
    ((2, 7), (7, 1))
]

for shape_a, shape_b in examples:
    A = torch.randn(*shape_a)
    B = torch.randn(*shape_b)
    C = A @ B
    print(f"{shape_a} @ {shape_b} -> {tuple(C.shape)}")

(5, 3) @ (3, 2) -> (5, 2)
(10, 8) @ (8, 4) -> (10, 4)
(2, 7) @ (7, 1) -> (2, 1)


# 44. Mini Exercise — Image Batch

Create:

```text
16 RGB images
224 × 224
```

Then:

1. Print the shape.
2. Extract the first image.
3. Print the first image shape.
4. Add a batch dimension back.
5. Verify that the final shape is `(1,3,224,224)`.

In [141]:
# Your solution

images = torch.rand(16, 3, 224, 224)

first = images[0]
first_batched = first.unsqueeze(0)

print("Batch:", images.shape)
print("First image:", first.shape)
print("First image batched:", first_batched.shape)

Batch: torch.Size([16, 3, 224, 224])
First image: torch.Size([3, 224, 224])
First image batched: torch.Size([1, 3, 224, 224])


# 45. Final Challenge 🚀 — Classification Tensor Operations

You are given:

```text
batch size = 8
number of classes = 5
```

Complete all of the following **without Python loops**:

1. Create random model scores of shape `(8,5)`.
2. Find the predicted class for every sample.
3. Create random target labels from `0` to `4`.
4. Compare predictions with targets.
5. Count correct predictions.
6. Calculate accuracy.
7. Create a Boolean mask of correct predictions.
8. Print all important shapes.

In [142]:
batch_size = 8
num_classes = 5

scores = torch.randn(batch_size, num_classes)

predictions = scores.argmax(dim=1)

targets = torch.randint(
    low=0,
    high=num_classes,
    size=(batch_size,)
)

correct_mask = predictions == targets
correct_count = correct_mask.sum()
accuracy = correct_mask.float().mean()

print("Scores shape      :", scores.shape)
print("Predictions shape :", predictions.shape)
print("Targets shape     :", targets.shape)
print("Correct mask      :", correct_mask)
print("Correct count     :", correct_count.item())
print("Accuracy          :", accuracy.item())

Scores shape      : torch.Size([8, 5])
Predictions shape : torch.Size([8])
Targets shape     : torch.Size([8])
Correct mask      : tensor([False,  True,  True, False, False, False, False,  True])
Correct count     : 3
Accuracy          : 0.375


# 46. Parameter Cheat Sheet

| Operation | Important Parameters | Purpose |
|---|---|---|
| `torch.matmul()` | `input`, `other` | Matrix/batched multiplication |
| `torch.sum()` | `dim`, `keepdim`, `dtype` | Sum values |
| `torch.mean()` | `dim`, `keepdim`, `dtype` | Average values |
| `torch.argmax()` | `dim`, `keepdim` | Largest-value index |
| `reshape()` | `shape` | Change shape |
| `flatten()` | `start_dim`, `end_dim` | Flatten dimensions |
| `transpose()` | `dim0`, `dim1` | Swap two dimensions |
| `permute()` | dimension order | Reorder all dimensions |
| `unsqueeze()` | `dim` | Add size-1 dimension |
| `squeeze()` | `dim` | Remove size-1 dimension |
| `cat()` | `tensors`, `dim` | Join existing dimension |
| `stack()` | `tensors`, `dim` | Create new dimension |
| `chunk()` | `chunks`, `dim` | Divide into chunks |
| `split()` | `split_size_or_sections`, `dim` | Divide by sizes |
| `clamp()` | `min`, `max` | Restrict values |
| `where()` | `condition`, `input`, `other` | Conditional selection |
| `randint()` | `low`, `high`, `size` | Random integers |

# 47. Common Mistakes

### ❌ Mistake 1 — Confusing multiplication

```python
A * B
```

→ element-wise

```python
A @ B
```

→ matrix multiplication

### ❌ Mistake 2 — Incorrect broadcasting

Compare dimensions from right to left.

### ❌ Mistake 3 — Invalid reshape

The number of elements must remain unchanged.

### ❌ Mistake 4 — Wrong `dim`

Always identify the axis you want to operate on.

### ❌ Mistake 5 — Accidental `squeeze()`

It can remove a batch dimension when its size is `1`.

### ❌ Mistake 6 — Missing batch dimension

```text
Image:       (C,H,W)
Image batch: (N,C,H,W)
```

### ❌ Mistake 7 — CPU/GPU mismatch

Tensors involved in an operation generally need compatible devices.

# 48. Knowledge Check

Before continuing, answer these without looking back:

1. What is broadcasting?
2. Why does `(2,3) + (3,)` work?
3. Why does `(2,3) + (2,2)` fail?
4. What is the difference between `*` and `@`?
5. What are the inner dimensions in matrix multiplication?
6. What does `dim=0` mean for a matrix?
7. What does `keepdim=True` change?
8. Difference between `cat()` and `stack()`?
9. Difference between `reshape()` and `view()`?
10. Why might `view()` fail after `transpose()`?
11. What does `permute()` do?
12. Why is `(N,C,H,W)` common for image batches?
13. What does `unsqueeze()` do?
14. Why can unrestricted `squeeze()` be dangerous?
15. What does `numel()` return?
16. What does `element_size()` return?
17. Why are dtype and device important?

# 49. Key Takeaways

The most important rules from this notebook:

### ⭐ Rule 1

```text
*  → element-wise multiplication
@  → matrix multiplication
```

### ⭐ Rule 2

```text
Broadcasting compares dimensions from right → left.
```

### ⭐ Rule 3

```text
reshape() cannot change the number of elements.
```

### ⭐ Rule 4

```text
dim tells PyTorch which axis to operate along.
```

### ⭐ Rule 5

```text
keepdim=True preserves the reduced dimension as size 1.
```

### ⭐ Rule 6

```text
cat() joins an existing dimension.
stack() creates a new dimension.
```

### ⭐ Rule 7

```text
(N,C,H,W) is a common image-batch layout.
```

### ⭐ Rule 8

```text
Always inspect shape, dtype, and device when debugging.
```

---

## ➡️ Next Notebook

### `03_Autograd_and_Gradients.ipynb`

The next notebook moves from tensor mathematics into the mechanism that makes neural-network training possible:

```text
Tensor
   ↓
requires_grad
   ↓
Computational Graph
   ↓
Gradient
   ↓
backward()
   ↓
Gradient Accumulation
   ↓
detach()
   ↓
no_grad()
   ↓
Backpropagation
```